## 2. Advanced Queries

## Import Required Libraries

In [1]:
import random
import pandas as pd
from faker import Faker
from datetime import datetime, timedelta

fake = Faker()

## Database Creation

In [2]:
import sqlite3
conn = sqlite3.connect("ecommerce.db")
print("Successfully Database Connected")
cursor = conn.cursor()

Successfully Database Connected


## Helper Function

In [3]:
def run_query(query):
   return pd.read_sql_query(query,conn)

## Table Creation

In [4]:
customers = pd.read_csv("customers.csv")
products = pd.read_csv("cleaned_products.csv")
orders = pd.read_csv("cleaned_orders.csv")
order_items = pd.read_csv("order_itmes.csv")

In [5]:
customers.to_sql("customers",conn,if_exists="replace",index=False)

products.to_sql("products",conn,if_exists="replace",index=False)

orders.to_sql("orders",conn,if_exists="replace",index=False)

order_items.to_sql("order_items",conn,if_exists="replace",index=False)

print("All tables loaded successfully.")

All tables loaded successfully.


### Q7. Running Totals with Window Functions

In [9]:
q7 = """with daily_sales as (select o.region_code,date(o.order_date) as order_date,
        sum(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)
        ) as daily_revenue
    from orders o
    join order_items oi
    on o.order_id = oi.order_id
    group by o.region_code, date(o.order_date))

select region_code, order_date, daily_revenue,
sum(daily_revenue) over(partition by region_code order by order_date) as running_total
from daily_sales
order by region_code, order_date;"""

df_q7 = run_query(q7)
df_q7

,region_code,order_date,daily_revenue,running_total
0,EAST,2024-07-13,29512.65,29512.65
1,EAST,2024-07-16,8031.84,37544.49
2,EAST,2024-08-23,12818.40,50362.89
3,EAST,2024-08-24,2768.22,53131.11
4,EAST,2024-09-14,9570.06,62701.17
...,...,...,...,...
295,WEST,2026-06-09,1160.28,569635.81
296,WEST,2026-06-10,9242.88,578878.69
297,WEST,2026-06-23,7878.36,586757.05
298,WEST,2026-06-30,3587.94,590344.99


### Q8. Ranking with DENSE_RANK

In [10]:
q8 = """select category,product_name,total_revenue,
dense_rank() over(partition by category order by total_revenue desc) as rank_in_category
from(select p.category,p.product_name,
sum(oi.quantity * oi.unit_price *(1 - oi.discount_percent/100.0)) as total_revenue
from products p
join order_items oi
on p.product_id=oi.product_id
group by p.category,p.product_name) as t;"""

df_q8 = run_query(q8)
df_q8

,category,product_name,total_revenue,rank_in_category
0,Books,Math Book,160868.72,1
1,Books,Dune,118065.90,2
2,Books,Harry Potter,92795.89,3
3,Books,Python Book,88133.34,4
4,Clothing,T-Shirt,199296.63,1
5,Clothing,Jeans,123106.23,2
6,Clothing,Handbag,113710.42,3
7,Clothing,Dress,102361.58,4
8,Clothing,Toy,84165.08,5
9,Clothing,Shirt,83723.94,6


### Q9. LAG/LEAD Analysis

In [11]:
q9 = """with cte as (select customer_id,order_date,
lag(order_date) over (partition by customer_id order by order_date) as previous_order_date
from orders)
select customer_id, order_date,previous_order_date,
round(julianday(order_date) - julianday(previous_order_date)) as days_gap
from cte;"""

df_q9 = run_query(q9)
df_q9

,customer_id,order_date,previous_order_date,days_gap
0,1.0,2024-12-22 13:46:09,None,NaN
1,10.0,2026-01-13 09:51:35,None,NaN
2,104.0,2024-12-07 14:16:39,None,NaN
3,104.0,2025-06-12 07:43:22,2024-12-07 14:16:39,187.0
4,104.0,2025-06-19 03:00:10,2025-06-12 07:43:22,7.0
...,...,...,...,...
495,UNKNOWN,2026-01-08 09:25:21,2025-12-31 19:45:13,8.0
496,UNKNOWN,2026-01-12 06:39:17,2026-01-08 09:25:21,4.0
497,UNKNOWN,2026-03-25 03:30:34,2026-01-12 06:39:17,72.0
498,UNKNOWN,2026-04-10 07:14:21,2026-03-25 03:30:34,16.0


### Q10. CTE with Multiple Levels

In [12]:
q10 = """with monthly_revenue as(
select o.customer_id,strftime('%Y-%m',o.order_date) as month,
sum(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)) revenue
from orders o
join order_items oi
on o.order_id=oi.order_id
group by o.customer_id, month),

customer_type as(
select *,
    case when revenue>10000 then 'High'
         when revenue>=5000 then 'Medium'
         else 'Low'
    end as category
from monthly_revenue)

select month,category, count(customer_id) customers
from customer_type
group by month, category
order by month;"""

df_q10 = run_query(q10)
df_q10

,month,category,customers
0,2024-07,High,3
1,2024-07,Low,5
2,2024-07,Medium,4
3,2024-08,High,6
4,2024-08,Low,6
...,...,...,...
70,2026-06,Low,7
71,2026-06,Medium,6
72,2026-07,High,1
73,2026-07,Low,1


### Q12. Year-over-Year Comparison

In [13]:
q12 = """with monthly_revenue as(
select strftime('%Y',order_date) year, strftime('%m',order_date) month,
sum(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)) revenue
from orders o
join order_items oi
on o.order_id=oi.order_id
group by year,month)

select year,month,revenue,
lag(revenue) over(
partition by month
order by year) prev_year_revenue,
round((revenue-lag(revenue) over(partition by month order by year))*100.0/
lag(revenue) over(partition by month order by year),2) yoy_growth_percent
from monthly_revenue;"""

df_q12 = run_query(q12)
df_q12

,year,month,revenue,prev_year_revenue,yoy_growth_percent
0,2025,01,167388.91,NaN,NaN
1,2026,01,123181.91,167388.91,-26.41
2,2025,02,59981.98,NaN,NaN
3,2026,02,102708.25,59981.98,71.23
4,2025,03,205125.91,NaN,NaN
5,2026,03,72215.93,205125.91,-64.79
6,2025,04,119440.56,NaN,NaN
7,2026,04,103771.91,119440.56,-13.12
8,2025,05,84167.27,NaN,NaN
9,2026,05,136469.28,84167.27,62.14


### Q13. First/Last Value Analysis

q13 =""" with cte as (
select o.customer_id, p.category,
row_number() over(partition by o.customer_id order by o.order_date) rn1,
row_number() over(partition by o.customer_id order by o.order_date desc) rn2
from orders o
join order_items oi on o.order_id = oi.order_id
join products p on oi.product_id = p.product_id
)

select a.customer_id,a.category AS first_category,b.category AS last_category,
case when a.category = b.category then 'No'
     else 'Yes'
end as category_shift
from cte a
join cte b
on a.customer_id = b.customer_id
and a.rn1 = 1
and b.rn2 = 1;"""

df_q13 = run_query(q13)
df_q13

### Q14. Cumulative Distribution

In [15]:
q14 = """with customer_revenue as(
select o.customer_id,
sum(oi.quantity *oi.unit_price *(1 - oi.discount_percent/100.0)) as revenue
from orders o
join order_items oi
on o.order_id = oi.order_id
group by o.customer_id
)

select customer_id,revenue,
sum(revenue) over(order by revenue desc) as cumulative_revenue,
round(sum(revenue) over(order by revenue desc) * 100.0 /
sum(revenue) over(),2) as cumulative_percent
from customer_revenue
order by revenue desc;"""

df_q14 = run_query(q14)
df_q14

,customer_id,revenue,cumulative_revenue,cumulative_percent
0,UNKNOWN,117005.78,117005.78,4.67
1,253.0,45437.65,162443.43,6.48
2,390.0,42537.44,204980.87,8.18
3,352.0,39431.85,244412.72,9.75
4,193.0,37579.22,281991.94,11.25
...,...,...,...,...
221,331.0,-930.58,2535067.48,101.11
222,107.0,-2101.98,2532965.50,101.03
223,88.0,-8047.86,2524917.64,100.71
224,114.0,-8434.80,2516482.84,100.37


### Q15. Complex CTE: Cohort Analysis

In [16]:
q15 = """with customer_cohort as(
select c.customer_id,strftime('%Y-%m', c.registration_date) as cohort_month,
strftime('%Y-%m', o.order_date) as order_month
from customers c
left join orders o
on c.customer_id = o.customer_id
)

select cohort_month,order_month,
count(distinct customer_id) as total_customers
from customer_cohort
group by cohort_month,order_month
order by cohort_month, order_month;"""

df_q15 = run_query(q15)
df_q15

,cohort_month,order_month,total_customers
0,2023-07,None,2
1,2023-07,2024-08,1
2,2023-07,2024-11,1
3,2023-07,2024-12,1
4,2023-07,2025-03,1
...,...,...,...
388,2026-06,2025-08,1
389,2026-06,2026-01,1
390,2026-06,2026-06,1
391,2026-07,2024-11,1
